# Temperature and fitness (stopping criterion fixed)

Data: `explore_temp_curves.csv` (bypass, mode=both, top90 + min 50 generations). Sweep of
**optimizer** (umda/emna/egna/keda), **fitness** (5), **Tc** (CPT softmax temperature) and
**Tu** (utility sigmoid temperature) ∈ {0.25,0.5,1,2,4}, and **% of rules** {10,20}.

Goal: show that recovery depends more on **temperature and fitness** than on the number of rules.

- **Section A** — all curves (runs) of one configuration.
- **Section B** — grid with selectable row/column axes (e.g. fitness × Tc).

**Marks:** multi-select (Ctrl/Shift) — choose thresholds (50/70/80/90/95/100).
**Reload** brings the latest (egna/keda run last).

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as W
from IPython.display import display

plt.rcParams.update({'font.size': 14, 'axes.titlesize': 15, 'axes.labelsize': 14,
                     'legend.fontsize': 12, 'xtick.labelsize': 12, 'ytick.labelsize': 12})

OUT_DIR = 'figures_temp'
MARKS = [(50, '*', '#d62728', 'top50'), (70, 'D', '#2ca02c', 'top70'),
         (90, 's', '#9467bd', 'top90'),
         (95, 'P', '#17becf', 'top95'), (100, 'X', '#8c564b', 'top100')]
THRS = [m[0] for m in MARKS]
AXES = ['optimizer', 'fitness_type', 'chance_temperature', 'utility_temperature', 'pct']
ALAB = {'optimizer': 'optimizer', 'fitness_type': 'fitness',
        'chance_temperature': 'Tc (CPT softmax)', 'utility_temperature': 'Tu (utility)',
        'pct': '% of rules'}
METRIC_LABEL = {'acc_best':  'Best-individual rule accuracy (%)',
                'acc_mean':  'Population-mean rule accuracy (%)',
                'acc_worst': 'Worst-individual rule accuracy (%)',
                'sat_acc_mean': 'Rule-satisfiers mean accuracy (%)'}

def _resolve(p):
    for base in ['', '..', '../..']:
        c = os.path.join(base, p) if base else p
        if os.path.exists(c):
            return c
    return p

def load_curves():
    path = _resolve('example/explore_temp_curves.csv')
    for _ in range(6):
        try:
            return pd.read_csv(path)
        except Exception:
            pass
    return pd.DataFrame()

CUR = load_curves()
print('rows:', len(CUR))
if len(CUR):
    print('optimizers:', sorted(CUR.optimizer.unique()), '| fitness:', sorted(CUR.fitness_type.unique()))
    print('Tc:', sorted(CUR.chance_temperature.unique()), '| Tu:', sorted(CUR.utility_temperature.unique()),
          '| % rules:', sorted(CUR.pct.unique()))

def uvals(df, ax):
    return sorted(df[ax].unique())

def crossing_gen(g, thr):
    h = g[g['pct_satisfiers'] >= thr]
    return int(h['generation'].iloc[0]) if len(h) else None

def rep_mean(d, col, freeze=False):
    if freeze:
        piv = d.pivot_table(index='generation', columns='rep', values=col)
        piv = piv.reindex(range(int(piv.index.min()), int(piv.index.max()) + 1)).ffill()
        return piv.mean(axis=1)
    return d.groupby('generation')[col].mean()

def rep_mean_std(d, col, freeze=False):
    if freeze:
        piv = d.pivot_table(index='generation', columns='rep', values=col)
        piv = piv.reindex(range(int(piv.index.min()), int(piv.index.max()) + 1)).ffill()
        return piv.mean(axis=1), piv.std(axis=1).fillna(0.0)
    grp = d.groupby('generation')[col]
    return grp.mean(), grp.std().fillna(0.0)

def median_cross(d, thr):
    gens = [crossing_gen(g.sort_values('generation'), thr) for _, g in d.groupby('rep')]
    gens = [x for x in gens if x is not None]
    return int(np.median(gens)) if gens else None

def place_marks(ax, series, gx_fn, sel, legend=False):
    sel = set(sel)
    for thr, mk, c, lb in MARKS:
        if thr not in sel:
            continue
        gx = gx_fn(thr)
        if gx is not None and gx in series.index:
            ax.plot(gx, series.loc[gx], mk, color=c, ms=13 if mk == '*' else 8,
                    markeredgecolor='k', label=(lb if legend else None), zorder=5)

In [ ]:
# ====================== SECTION A — all curves of ONE config ======================
_A = {'fig': None, 'busy': 0}
a_opt = W.Dropdown(description='Optimizer')
a_fit = W.Dropdown(description='Fitness')
a_tc = W.Dropdown(description='Tc (CPT)')
a_tu = W.Dropdown(description='Tu (util.)')
a_pct = W.Dropdown(description='% rules')
a_met = W.Dropdown(description='Metric', options=['acc_best', 'acc_mean', 'sat_acc_mean'])
a_rep = W.Dropdown(description='Repetition', options=['All'])
a_marks = W.SelectMultiple(description='marks', options=THRS, value=tuple(THRS), rows=6)
a_freeze = W.Checkbox(value=True, description='freeze after stop')
a_yr = W.FloatRangeSlider(value=[0, 100], min=0, max=100, step=5, description='Y axis',
                          readout_format='.0f', continuous_update=False)
a_fname = W.Text(description='File', placeholder='auto')
a_png = W.Button(description='Export PNG', button_style='success', icon='download')
a_reload = W.Button(description='Reload', icon='refresh')
a_out = W.Output(); a_msg = W.Output()

def _a_pop(*_):
    if not len(CUR):
        return
    _A['busy'] += 1
    try:
        a_opt.options = uvals(CUR, 'optimizer')
        if a_opt.value not in a_opt.options:
            a_opt.value = 'umda' if 'umda' in a_opt.options else a_opt.options[0]
        a_fit.options = uvals(CUR, 'fitness_type')
        a_tc.options = uvals(CUR, 'chance_temperature')
        a_tu.options = uvals(CUR, 'utility_temperature')
        a_pct.options = uvals(CUR, 'pct')
        for w, dv in [(a_fit, 'binary'), (a_tc, 1.0), (a_tu, 1.0), (a_pct, 10)]:
            if w.value not in w.options:
                w.value = dv if dv in w.options else w.options[0]
        sub = CUR[(CUR.optimizer == a_opt.value) & (CUR.fitness_type == a_fit.value)
                  & (CUR.chance_temperature == a_tc.value) & (CUR.utility_temperature == a_tu.value)
                  & (CUR.pct == a_pct.value)]
        a_rep.options = ['All'] + sorted(int(r) for r in sub.rep.unique())
    finally:
        _A['busy'] -= 1

def _a_draw(*_):
    if _A['busy']:
        return
    with a_out:
        a_out.clear_output(wait=True)
        d = CUR[(CUR.optimizer == a_opt.value) & (CUR.fitness_type == a_fit.value)
                & (CUR.chance_temperature == a_tc.value) & (CUR.utility_temperature == a_tu.value)
                & (CUR.pct == a_pct.value)]
        if not len(d):
            print('No data for this config (not computed yet?).'); return
        m = a_met.value
        fig, ax = plt.subplots(figsize=(11, 6.4))
        if a_rep.value == 'All':
            for _, g in d.groupby('rep'):
                g = g.sort_values('generation')
                ax.plot(g['generation'], g[m], color='#9bb3d4', lw=1, alpha=.8)
            mean = rep_mean(d, m, a_freeze.value)
            ax.plot(mean.index, mean.values, color='#1f4e79', lw=3, label='mean over runs')
            place_marks(ax, mean, lambda thr: median_cross(d, thr), a_marks.value, legend=True)
            sub_t = f'{d.rep.nunique()} runs + mean'
        else:
            g = d[d.rep == int(a_rep.value)].sort_values('generation').set_index('generation')[m]
            ax.plot(g.index, g.values, color='#1f4e79', lw=2.5, marker='o', ms=3, label=f'run {a_rep.value}')
            gg = d[d.rep == int(a_rep.value)].sort_values('generation')
            place_marks(ax, g, lambda thr: crossing_gen(gg, thr), a_marks.value, legend=True)
            sub_t = f'run {a_rep.value}'
        ax.set_xlabel('Generation'); ax.set_ylabel(METRIC_LABEL.get(m, m))
        ax.set_ylim(a_yr.value); ax.grid(alpha=.3)
        ax.set_title(f'{a_opt.value.upper()} \u00b7 {a_fit.value} fitness \u00b7 Tc={a_tc.value} Tu={a_tu.value} '
                     f'\u00b7 {a_pct.value}% of rules \u2014 {sub_t}')
        ax.legend(loc='lower right')
        plt.tight_layout(); _A['fig'] = fig; plt.show(); plt.close(fig)

def _a_export(_):
    with a_msg:
        a_msg.clear_output(wait=True)
        if _A['fig'] is None:
            print('Draw something first.'); return
        os.makedirs(OUT_DIR, exist_ok=True)
        n = a_fname.value.strip() or f'temp_{a_opt.value}_{a_fit.value}_Tc{a_tc.value}_Tu{a_tu.value}_{a_pct.value}pct'
        n = n if n.lower().endswith('.png') else n + '.png'
        p = os.path.join(OUT_DIR, n); _A['fig'].savefig(p, dpi=150, bbox_inches='tight')
        print('Saved:', os.path.abspath(p))

def _a_reload(_):
    global CUR
    CUR = load_curves()
    with a_msg:
        a_msg.clear_output(wait=True); print(f'Reloaded: {len(CUR)} rows.')
    _a_pop(); _a_draw()

for w in [a_opt, a_fit, a_tc, a_tu, a_pct]:
    w.observe(_a_pop, names='value')
for w in [a_opt, a_fit, a_tc, a_tu, a_pct, a_met, a_rep, a_marks, a_freeze, a_yr]:
    w.observe(_a_draw, names='value')
a_png.on_click(_a_export); a_reload.on_click(_a_reload)
_a_pop(); _a_draw()
display(W.VBox([W.HBox([a_opt, a_fit, a_pct]),
                W.HBox([a_tc, a_tu, a_met]),
                W.HBox([a_rep, a_freeze, a_marks]),
                W.HBox([a_yr]),
                W.HBox([a_fname, a_png, a_reload]), a_msg]), a_out)

In [ ]:
# ====================== SECTION B — grid: choose row & column axes ======================
_B = {'fig': None, 'busy': 0}
b_row = W.Dropdown(description='Rows', options=AXES, value='fitness_type')
b_col = W.Dropdown(description='Columns', options=AXES, value='chance_temperature')
b_met = W.Dropdown(description='Metric', options=['acc_mean', 'acc_best', 'sat_acc_mean'])
b_disp = W.Dropdown(description='Spread',
                    options=['\u00b11\u03c3 (band)', 'best/worst (dotted)', 'dotted + \u03c3', 'none'])
b_marks = W.SelectMultiple(description='marks', options=THRS, value=tuple(THRS), rows=6)
b_freeze = W.Checkbox(value=True, description='freeze')
b_yr = W.FloatRangeSlider(value=[0, 100], min=0, max=100, step=5, description='Y axis',
                          readout_format='.0f', continuous_update=False)
b_fix = {ax: W.Dropdown(description=ALAB[ax]) for ax in AXES}
b_fname = W.Text(description='File', placeholder='auto')
b_png = W.Button(description='Export PNG', button_style='success', icon='download')
b_reload = W.Button(description='Reload', icon='refresh')
b_out = W.Output(); b_msg = W.Output()

def _b_pop(*_):
    if not len(CUR):
        return
    _B['busy'] += 1
    try:
        for ax in AXES:
            opts = uvals(CUR, ax)
            b_fix[ax].options = opts
            if b_fix[ax].value not in opts:
                b_fix[ax].value = opts[0]
    finally:
        _B['busy'] -= 1

def _b_draw(*_):
    if _B['busy']:
        return
    with b_out:
        b_out.clear_output(wait=True)
        rd, cd, m, sel = b_row.value, b_col.value, b_met.value, set(b_marks.value)
        if rd == cd:
            print('Rows and columns must be different axes.'); return
        fixed = [a for a in AXES if a not in (rd, cd)]
        d = CUR
        for a in fixed:
            d = d[d[a] == b_fix[a].value]
        if not len(d):
            print('No data for the fixed axes.'); return
        rvals, cvals = uvals(d, rd), uvals(d, cd)
        disp = b_disp.value
        fig, axes = plt.subplots(len(rvals), len(cvals),
                                 figsize=(3.3 * len(cvals) + 1, 2.9 * len(rvals) + 1),
                                 sharey=True, sharex=True, squeeze=False)
        for i, rv in enumerate(rvals):
            for j, cv in enumerate(cvals):
                ax = axes[i][j]
                dd = d[(d[rd] == rv) & (d[cd] == cv)]
                if len(dd):
                    mean, std = rep_mean_std(dd, m, b_freeze.value)
                    x = mean.index.values
                    if disp in ('\u00b11\u03c3 (band)', 'dotted + \u03c3'):
                        ax.fill_between(x, mean - std, mean + std, color='#4C72B0', alpha=.18)
                    if disp in ('best/worst (dotted)', 'dotted + \u03c3'):
                        ax.plot(x, rep_mean(dd, 'acc_best', b_freeze.value).values, ':', color='#55A868', lw=1.2)
                        ax.plot(x, rep_mean(dd, 'acc_worst', b_freeze.value).values, ':', color='#C44E52', lw=1.2)
                    ax.plot(x, mean.values, color='#1f4e79', lw=2)
                    place_marks(ax, mean, lambda thr: median_cross(dd, thr), sel)
                ax.set_ylim(b_yr.value); ax.grid(alpha=.3)
                if i == 0:
                    ax.set_title(f'{ALAB[cd]} = {cv}', fontsize=12, fontweight='bold')
                if j == 0:
                    ax.set_ylabel(f'{ALAB[rd]} = {rv}\n{METRIC_LABEL.get(m, m)}', fontsize=11)
                if i == len(rvals) - 1:
                    ax.set_xlabel('Generation')
        fixtxt = ', '.join(f'{ALAB[a]}={b_fix[a].value}' for a in fixed)
        fig.suptitle(f'{ALAB[rd]} (rows) \u00d7 {ALAB[cd]} (columns)   |   fixed: {fixtxt}',
                     y=1.0, fontweight='bold')
        plt.tight_layout(); _B['fig'] = fig; plt.show(); plt.close(fig)

def _b_export(_):
    with b_msg:
        b_msg.clear_output(wait=True)
        if _B['fig'] is None:
            print('Draw something first.'); return
        os.makedirs(OUT_DIR, exist_ok=True)
        n = b_fname.value.strip() or f'tempgrid_{b_row.value}_x_{b_col.value}_{b_met.value}'
        n = n if n.lower().endswith('.png') else n + '.png'
        p = os.path.join(OUT_DIR, n); _B['fig'].savefig(p, dpi=150, bbox_inches='tight')
        print('Saved:', os.path.abspath(p))

def _b_reload(_):
    global CUR
    CUR = load_curves()
    with b_msg:
        b_msg.clear_output(wait=True); print(f'Reloaded: {len(CUR)} rows.')
    _b_pop(); _b_draw()

_b_pop()
for w in [b_row, b_col, b_met, b_disp, b_marks, b_freeze, b_yr] + list(b_fix.values()):
    w.observe(_b_draw, names='value')
b_png.on_click(_b_export); b_reload.on_click(_b_reload)
_b_draw()
display(W.VBox([W.HBox([b_row, b_col, b_met]),
                W.HBox([b_disp, b_freeze, b_marks]),
                W.HBox(list(b_fix.values())),
                W.HBox([b_yr]),
                W.HBox([b_fname, b_png, b_reload]), b_msg]), b_out)